In [27]:
from itertools import product
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import normalize
from scipy import sparse

pool = pd.read_parquet("data/pool_splits.parquet")
AAS = "ACDEFGHIKLMNPQRSTVWY"

In [15]:
seqs = pool['protein_stripped']
labels = pool['label']

In [22]:
final = {}
for protein in AAS:
    final[protein] =[]

In [23]:
for idx, seq in seqs.items():
    for protein in AAS:
        final[protein].append(seq.count(protein)/len(seq))

In [24]:
comp = pd.DataFrame(final, index=pool.index)
comp

,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
0,0.083516,0.010989,0.070330,0.063736,0.043956,0.087912,0.024176,0.057143,0.057143,0.085714,0.032967,0.043956,0.030769,0.032967,0.041758,0.052747,0.076923,0.070330,0.008791,0.024176
1,0.017241,0.017241,0.034483,0.064655,0.051724,0.021552,0.030172,0.077586,0.146552,0.142241,0.038793,0.060345,0.017241,0.051724,0.064655,0.056034,0.025862,0.025862,0.012931,0.043103
2,0.072351,0.002584,0.069767,0.098191,0.041344,0.059432,0.028424,0.062016,0.054264,0.080103,0.038760,0.025840,0.023256,0.020672,0.093023,0.046512,0.031008,0.108527,0.005168,0.038760
3,0.032558,0.000000,0.046512,0.083721,0.018605,0.046512,0.051163,0.111628,0.093023,0.097674,0.027907,0.074419,0.013953,0.013953,0.046512,0.051163,0.060465,0.055814,0.009302,0.065116
4,0.119048,0.013228,0.050265,0.071429,0.034392,0.105820,0.010582,0.050265,0.031746,0.121693,0.013228,0.039683,0.055556,0.026455,0.071429,0.047619,0.029101,0.074074,0.023810,0.010582
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69415,0.064671,0.005988,0.057485,0.106587,0.027545,0.069461,0.023952,0.069461,0.070659,0.080240,0.045509,0.047904,0.020359,0.049102,0.064671,0.039521,0.051497,0.074251,0.002395,0.028743
69416,0.063415,0.007317,0.036585,0.097561,0.041463,0.075610,0.019512,0.051220,0.051220,0.109756,0.039024,0.034146,0.058537,0.007317,0.065854,0.060976,0.058537,0.102439,0.004878,0.014634
69417,0.052795,0.012422,0.040373,0.049689,0.065217,0.065217,0.021739,0.111801,0.093168,0.099379,0.012422,0.065217,0.015528,0.034161,0.031056,0.086957,0.040373,0.040373,0.006211,0.055901
69418,0.074359,0.010256,0.041026,0.082051,0.025641,0.074359,0.028205,0.102564,0.053846,0.071795,0.030769,0.071795,0.028205,0.030769,0.048718,0.051282,0.056410,0.084615,0.007692,0.025641


In [25]:
print(comp.shape)
print(comp.sum(axis=1).describe())
print(comp.isna().sum().sum())
print((comp.index == pool.index).all())

(69420, 20)
count    6.942000e+04
mean     1.000000e+00
std      1.314677e-16
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
0
True


In [26]:
comp.to_parquet("data/features_comp.parquet")

In [28]:
kmers = ["".join(p) for p in product(AAS, repeat=3)]

vec = CountVectorizer(
    analyzer="char",
    ngram_range=(3, 3),
    lowercase=False,
    vocabulary=kmers,
)
X_counts = vec.transform(pool["protein_stripped"])
X_kmer = normalize(X_counts, norm="l1")

In [29]:
print(len(kmers), kmers[:3])
print(X_kmer.shape)
print(X_kmer.nnz)
print(X_kmer.sum(axis=1).min(), X_kmer.sum(axis=1).max())

8000 ['AAA', 'AAC', 'AAD']
(69420, 8000)
19233985
0.9999999999999993 1.0000000000000007


In [30]:
sparse.save_npz("data/features_kmer.npz", X_kmer)